# 01 - Initial data inspection
FlatFair. Inspects every dataset currently in `data/raw/`. Household income and transit files are **not yet present** (see last cell).

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd, numpy as np
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))
from paths import *
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
def inspect(df, name, cat_cols=()):
    print(f"=== {name} ===  shape={df.shape}  memory={df.memory_usage(deep=True).sum()/1e6:.1f} MB")
    print(df.dtypes.to_string()); print("\nHEAD"); print(df.head(3).to_string()); print("\nTAIL"); print(df.tail(3).to_string())
    miss = df.isna().sum().to_frame("n_missing").assign(pct=lambda x: x.n_missing/len(df)*100)
    print("\nMISSING\n", miss[miss.n_missing>0] if miss.n_missing.any() else "none")
    print("\nEXACT DUPLICATES:", df.duplicated().sum())
    print("\nUNIQUE COUNTS\n", df.nunique().to_string())
    print("\nNUMERIC SUMMARY\n", df.describe().T.to_string())
hdb = pd.read_csv(HDB_RAW)
inspect(hdb, "HDB resale (raw)")

In [ ]:
print("Month range:", hdb.month.min(), "->", hdb.month.max())
print(hdb.month.str[:4].value_counts().sort_index().to_string())
for c in ["town","flat_type","flat_model"]: print(f"\n{c}:", hdb[c].value_counts().to_dict())
print("\nlease_commence_date:", hdb.lease_commence_date.min(), hdb.lease_commence_date.max())
print("storey_range levels:", sorted(hdb.storey_range.unique()))
print("remaining_lease formats:\n", hdb.remaining_lease.str.replace(r"\d+","N",regex=True).value_counts().to_string())
print("\nMost recent month is partial?", hdb.month.value_counts().sort_index().tail(3).to_dict())

**HDB observations**
- No missing values. 318 exact duplicate rows exist; the file has no unit number or sale day so identical sales can be genuine. They are **flagged, not removed**.
- `remaining_lease` has three text shapes (`N years N months`, `N years`, `N years 1 month`), so the parser must handle singular/missing parts.
- `flat_model` mixes casing/labels (e.g. `Multi Generation`, `3Gen`, `2-room`, `Premium Apartment Loft`).
- The latest month(s) may be partial; monthly trends should treat them with care.

In [ ]:
dup = hdb[hdb.duplicated(keep=False)]
print(len(dup), "rows involved in duplicate groups;", "groups of size:", dup.groupby(list(hdb.columns)).size().value_counts().to_dict())
print(dup.groupby("town").size().sort_values(ascending=False).head(5).to_dict())

In [ ]:
pop = pd.read_csv(POP_RAW)
inspect(pop, "Population 2020 (raw)")
print("\n'-' count by column:\n", (pop=="-").sum().to_string())

In [ ]:
from data_cleaning import clean_population, population_reconciliation
full, pa = clean_population(pop)
print(full.level.value_counts().to_string())
rec = population_reconciliation(full)
print(rec.diff_pct.describe().round(2).to_string())
print("\nLargest gaps (rounding to nearest 10 + suppressed subzones):")
print(rec.reindex(rec.diff_pct.abs().sort_values(ascending=False).index).head(6).to_string())

**Population observations**
- 1 national row, 55 planning-area totals (including source typo `Changi- Total`), rest subzones. `"-"` cells are suppressed values -> NaN, not zero.
- Counts are rounded to the nearest 10, so planning-area totals reconcile with the sum of subzones only approximately; gaps appear where subzones are suppressed or area totals are blank.
- Whampoa does **not** appear as a subzone in this file (relevant to `KALLANG/WHAMPOA`).

In [ ]:
inc_raw = pd.read_csv(INCOME_RAW)
inspect(inc_raw, "Household income by planning area (raw)")
from household_income import clean_income, income_features, BAND_COLS
inc, inc_long = clean_income(inc_raw)
print("\nOrientation: one ROW per planning area, one COLUMN per income band (wide); 'Total' row dropped, 'Others' kept.")
print("Income bands (ordered):", BAND_COLS)
print("NoEmployedPerson households are outside the bands. band_sum_check (bands + no-employed - total) range:", inc.band_sum_check.min(), "to", inc.band_sum_check.max(), "(rounding)")
print("Suppressed/missing cells:", inc_raw.isna().sum().sum())
print("Planning areas in income file but not in population file:", sorted(set(inc.planning_area) - set(pd.read_csv(POP_RAW).Number.str.replace(" - Total","")))) 

**Income observations**
- Wide format, 30 areas + `Others`, no suppressed cells. Counts reconcile to `Total` within +-3 households (rounding).
- Income is household income **from work**; households with no employed person (6-27% by area) have no band and are reported separately.
- Top band `$20,000 and over` is open-ended: the median cannot be interpolated where it falls there (Bukit Timah, River Valley, Tanglin).

In [ ]:
import json
for name, p in [("MRT/LTA station EXITS", MRT_RAW), ("LTA bus stops", BUS_RAW)]:
    g = json.load(open(p)); f = g["features"]
    print(f"{name}: {len(f)} features; geometry types {set(x['geometry']['type'] for x in f)}; properties {list(f[0]['properties'])}")
    xs = np.array([x["geometry"]["coordinates"] for x in f]); print("  lon range", xs[:,0].min().round(3), xs[:,0].max().round(3), "lat range", xs[:,1].min().round(3), xs[:,1].max().round(3))
from geographic_features import load_mrt_stations, load_bus_stops
m = load_mrt_stations(); b = load_bus_stops()
print("Distinct MRT/LRT stations:", m.station_name.nunique(), "| exits:", len(m), "| duplicate bus stop numbers:", b.bus_stop_name.duplicated().sum())
print("Bus stops carry only a stop number (no name); MRT file is exit-level so exits are collapsed to stations for counts.")